# 🎯 Centralized Deep Learning Training Pipeline (Scenario E1)
## Empirical Performance Upper Bound Ceiling on CICIoT2023
### Master's Thesis: *Xây dựng và đánh giá prototype hệ thống phát hiện xâm nhập IoT sử dụng Federated Learning trong môi trường dữ liệu non-IID*
**Author:** Nguyễn Việt Kỳ Quân (2026) | **Institution:** HCMUT - VNU-HCM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NVKQ2022/FedLearning/blob/main/notebooks/centralized_training_pipeline.ipynb)

---

### 🎯 Purpose in Thesis
In Federated Learning research, **Scenario E1 (Centralized Training)** serves as the critical **empirical performance upper bound ceiling**. By pooling all training data in a single central repository without edge privacy constraints or non-IID partition boundaries, this benchmark determines the highest achievable detection accuracy, **Macro-F1**, and **Minority Recall** for the `TabularIoTMLP` architecture.

All subsequent decentralized experiments (Scenarios E2 through E5) are evaluated against this ceiling to measure the exact cost of statistical heterogeneity.

---
### 🛠️ Step 1: Environment Setup & Hardware Acceleration
We verify GPU acceleration, pull the latest code in Google Colab, and lock the deterministic random seed (`seed=42`).

In [ ]:
import os, sys, subprocess, warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Compute Device: {device} | PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'   Active GPU: {torch.cuda.get_device_name(0)}')

# Colab setup
if os.path.exists('/content'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass
    repo_dir = '/content/FedLearning'
    if not os.path.exists(repo_dir):
        subprocess.run(['git', 'clone', 'https://github.com/NVKQ2022/FedLearning.git', repo_dir], check=True)
    os.chdir(repo_dir)
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
elif os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from src.utils.seed import set_seed
set_seed(42)
print('✅ Deterministic random seed locked at 42.')

---
### ⚙️ Step 2: Unified Hyperparameter Configuration
All hyperparameters for Scenario E1 are defined here using `ExperimentConfig`.

In [ ]:
from src.utils.config import ExperimentConfig

CONFIG = ExperimentConfig(
    experiment_name="centralized_e1_baseline",
    seed=42,
    
    # Dataset & Preprocessing
    sample_size=100000,                 # 100k for rapid execution, or None for full 5.11M rows
    test_size=0.2,                      # 20% holdout test set (unseen)
    val_size=0.1,                       # 10% validation set
    scaler_type="robust",               # 'robust' (median/IQR) to handle heavy-tailed features
    batch_size=128,
    
    # Architecture & Regularization
    hidden_dims=(128, 64),              # Lightweight edge MLP backbone (~15k params)
    dropout_rate=0.2,
    
    # Loss Function (Class Imbalance Mitigation)
    loss_type="focal_loss",             # 'focal_loss' to suppress majority DDoS gradients
    class_weight_strategy="balanced",   # Inverse frequency class weighting
    focal_gamma=2.0,
    
    # Optimization & Training
    optimizer_type="adamw",
    learning_rate=1e-3,
    weight_decay=1e-4,
    epochs=12,
    patience=3,
    monitor_metric="val_loss",
    max_grad_norm=5.0
)

CONFIG.display()

---
### 💾 Step 3: Leak-Free Preprocessing & Class Weights
We locate the dataset and apply `load_and_preprocess_ciciot2023`. Preprocessing statistics (imputation medians, `RobustScaler`, label encodings) are strictly fitted on the training split to guarantee zero data leakage.

In [ ]:
from src.data.preprocess import load_and_preprocess_ciciot2023, compute_balanced_class_weights

# Locate dataset CSV
candidate_paths = [
    CONFIG.gdrive_csv,
    CONFIG.alt_gdrive_csv,
    CONFIG.target_csv_path,
    'datasets/CICIOT2023/merged_CICIOT2023_data.csv',
    'datasets/merged_CICIOT2023_data.csv',
]
csv_path = next((p for p in candidate_paths if os.path.exists(p) and os.path.getsize(p) > 10*1024*1024), None)
if not csv_path:
    csv_path = CONFIG.target_csv_path

print(f"📁 Loading and preprocessing from: {csv_path}")
data = load_and_preprocess_ciciot2023(
    csv_path=csv_path,
    test_size=CONFIG.test_size,
    val_size=CONFIG.val_size,
    sample_size=CONFIG.sample_size,
    scaler_type=CONFIG.scaler_type,
    batch_size=CONFIG.batch_size,
    random_state=CONFIG.seed
)

class_weights = compute_balanced_class_weights(
    y=data['y_train'],
    num_classes=data['num_classes'],
    strategy=CONFIG.class_weight_strategy
)
class_names = data['class_names']
minority_classes = ['Web-based', 'Brute-force']

print(f"✅ Training set:   {len(data['X_train']):,} samples ({len(data['train_loader']):,} batches)")
print(f"✅ Validation set: {len(data['X_val']):,} samples")
print(f"✅ Holdout Test:   {len(data['X_test']):,} samples")
print(f"Classes ({data['num_classes']}): {class_names}")

---
### 🧠 Step 4: Model Architecture & Training Execution
We train `TabularIoTMLP` with `MultiClassFocalLoss` and `CentralizedTrainer`.

In [ ]:
from src.models.mlp import TabularIoTMLP
from src.losses.focal_loss import build_loss_function
from src.optimizers.optimizer import build_optimizer
from src.training.trainer import CentralizedTrainer

# 1. Initialize Model Backbone
model = TabularIoTMLP(
    input_dim=data['input_dim'],
    hidden_dims=CONFIG.hidden_dims,
    num_classes=data['num_classes'],
    dropout_rate=CONFIG.dropout_rate
).to(device)

# 2. Build Loss Function & Optimizer
criterion = build_loss_function(
    loss_type=CONFIG.loss_type,
    class_weights=class_weights if CONFIG.class_weight_strategy != 'none' else None,
    gamma=CONFIG.focal_gamma,
    device=device
)
optimizer = build_optimizer(
    model=model,
    optimizer_type=CONFIG.optimizer_type,
    lr=CONFIG.learning_rate,
    weight_decay=CONFIG.weight_decay
)

# 3. Execute Centralized Training with Early Stopping
trainer = CentralizedTrainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    max_grad_norm=CONFIG.max_grad_norm
)

print(f"Starting Centralized Training ({CONFIG.epochs} max epochs, patience={CONFIG.patience})...")
history = trainer.fit(
    train_loader=data['train_loader'],
    val_loader=data['val_loader'],
    epochs=CONFIG.epochs,
    patience=CONFIG.patience,
    monitor=CONFIG.monitor_metric,
    save_top_k=CONFIG.save_top_k,
    verbose=CONFIG.verbose
)

---
### 📈 Step 5: Training Convergence Diagnostics
We inspect loss and accuracy learning curves across epochs.

In [ ]:
from src.visualization import plot_learning_curves

os.makedirs('reports/figures', exist_ok=True)
fig_curves = plot_learning_curves(
    history=history,
    title=f"Scenario E1: Centralized Training Convergence",
    loss_name=f"{CONFIG.loss_type.capitalize()} Loss",
    save_path="reports/figures/centralized_learning_curves.png",
    show=True
)

---
### 🏆 Step 6: Holdout Evaluation & Upper Bound Verification
We evaluate the best model checkpoint on the holdout test set (unseen, zero data leakage), reporting:
1. **Overall Accuracy & Macro-F1**
2. **Minority Recall** (`Web-based` and `Brute-force` attacks)
3. **Normalized Confusion Matrix**

In [ ]:
from src.evaluation.evaluator import evaluate_comprehensive
from src.visualization import plot_confusion_matrix, plot_minority_recall

# Evaluate on test set
test_results = evaluate_comprehensive(
    model=model,
    dataloader=data['test_loader'],
    class_names=class_names,
    minority_classes=minority_classes,
    device=device
)

print('=' * 65)
print('🏆 SCENARIO E1 (CENTRALIZED UPPER BOUND) FINAL EVALUATION')
print('=' * 65)
print(f"Test Accuracy:    {test_results['accuracy'] * 100:.2f}%")
print(f"Macro-F1 Score:   {test_results['macro_f1'] * 100:.2f}%")
print(f"Weighted-F1:      {test_results['weighted_f1'] * 100:.2f}%")
print(f"Macro-Precision:  {test_results['macro_precision'] * 100:.2f}%")
print(f"Macro-Recall:     {test_results['macro_recall'] * 100:.2f}%")
print('-' * 65)
print('Minority Class Recall (Stealth Infiltration Intrusions):')
for cls, rec in test_results['minority_recall'].items():
    print(f"   • {cls:<15}: {rec * 100:.2f}%")
print('=' * 65)

# Render Confusion Matrix & Minority Recall charts
plot_confusion_matrix(
    cm=test_results['confusion_matrix'],
    class_names=class_names,
    title="Scenario E1: Normalized Confusion Matrix",
    save_path="reports/figures/centralized_confusion_matrix.png",
    show=True
)

plot_minority_recall(
    minority_data=test_results['minority_recall'],
    title="Scenario E1: Minority Intrusion Recall",
    save_path="reports/figures/centralized_minority_recall.png",
    show=True
)

---
### 💾 Step 7: Export Checkpoints & Metrics Summary
We persist the trained model checkpoint (`.pth`), preprocessor (`.pkl`), and metrics report (`reports/centralized_e1_metrics.json`).

In [ ]:
import json, shutil

os.makedirs('checkpoints', exist_ok=True)
os.makedirs('reports', exist_ok=True)

# 1. Save model checkpoint & preprocessor
ckpt_path = 'checkpoints/centralized_e1_best.pth'
torch.save(model.state_dict(), ckpt_path)

preprocessor_path = 'checkpoints/preprocessor.pkl'
data['preprocessor'].save(preprocessor_path)

# 2. Export metrics JSON
metrics_export = {k: v for k, v in test_results.items() if k != 'confusion_matrix'}
metrics_file = 'reports/centralized_e1_metrics.json'
with open(metrics_file, 'w') as f:
    json.dump(metrics_export, f, indent=4)

print(f"✅ Model Checkpoint saved: {ckpt_path}")
print(f"✅ Preprocessor saved:     {preprocessor_path}")
print(f"✅ Metrics Report saved:   {metrics_file}")

# 3. Backup to Google Drive if mounted
if os.path.exists('/content/drive/MyDrive'):
    gdrive_dir = '/content/drive/MyDrive/NguyenVietKyQuanKLTN/checkpoints'
    os.makedirs(gdrive_dir, exist_ok=True)
    shutil.copy(ckpt_path, os.path.join(gdrive_dir, os.path.basename(ckpt_path)))
    print(f"✅ Checkpoint backed up to Google Drive: {gdrive_dir}")

print('\n🎉 Scenario E1 Complete: Centralized upper-bound benchmark established!')